# CS236781: Deep Learning on Computational Accelerators
# Final Project

Faculty of Computer Science, Technion.

Submitted by:

| #       |              Name |             Id |             email                  |
|---------|-------------------|----------------|----------------------------------- |
|Student 1|  Daniel Elgarici  |   305341828    | elgarici-dan@campus.technion.ac.il |
|Student 2|  Tal Benjo        |   318655701    | tal.benjo@campus.technion.ac.il    |

# Introduction

In this project we strive to reproduce and rexamine key results from the following article:

[No Data, No Optimization: A Lightweight Method to Disrupt Neural Networks with Sign-Flips (Galil et al., 2025)](https://arxiv.org/abs/2502.07408)

The main point of the article is that very few, intelligent changes to to the model's parameters can cause great damage to its accuracy. The method of choice that is refined in the article is to select the highest magnitude parameters and simply flip their sign bit. This method is then evolved over the article, and we'll discuss the different iterations in our work as well.

The article focused its efforts into visual models and CNNs in particular, we will be delving into the realm of LLMs focusing on BERT. We'll reexamine these methods on the dynamics of LLMs to see if they work as well, and perhaps suggest other methods and evaluate them.

## Reproducing a key result from the article
Our first order of business is to implement the attack strategies discussed in the article, see if we can reproduce the findings in the paper, and by doing so validate both our implementation and the results. One of the most staggering findings in the article is that even with a minimal change of sign bits you can substantially damage the model and alter its output. A concrete result discussed in the article was targeting ShuffleNetV2 with DNL attack, resullting in >99% Accuracy Reduction. Note that accuracy reduction is computed as follows:

$AR(k) = \frac{Acc(original)-Acc(altered_k)}{Acc(original)}$

where k is the number of of sign bits flipped, original is the model with itis original paramteres and altered_k is the model after flipping k bits. 
We reran the very same experiment to test our implementation and verify the results.

### Setup
To evaluate our methods we will need both a model and a data set. We opted to go with ShuffleNet since its both accessible and displayed impressive results in the article. The model we are taking is ShuffleNet and the dataset we were evaluating on is ImageNet similarly to the article.

In this section we define all the utilites we will need and load the model and the the dataset, including the very straight forward scoring function for collecting candidates for sign flips:

$Score(\theta_i) = |\theta_i| $

As elaborated in the article, in this first phase we simply take the highest magnitude paramters.


In [1]:
# If needed:
# !pip install torch torchvision --quiet

import torch, torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

VAL_DIR   = "/home/elgarici-dan/project_deep/dataset/val_final"  # organized as val/<synset>/*.JPEG
BATCH_SIZE = 256
NUM_WORKERS = 8
RESIZE, CROP = 256, 224

tfm = transforms.Compose([
    transforms.Resize(RESIZE),
    transforms.CenterCrop(CROP),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
]) # The normalization values are based on the statistics of the training data
# We rescale, crop and normalize to ensure the model seens inputs in the same distribution it was trained on

val_ds = datasets.ImageFolder(VAL_DIR, transform=tfm)

In [2]:
@torch.no_grad()
def calc_acc(model, loader, device=DEVICE):
    model.eval()
    correct = total = 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        pred = model(x).argmax(1)
        correct += (pred == y).sum().item()
        total   += y.numel()
    return 100. * correct / max(1, total)

# Extracts first L (learnable!!!) layers for CNN
def first_L_param_layers_CNN(model, L):
    layers = []
    for m in model.modules():
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            layers.append(m)
    return layers[:L]
    
def collect_dnl_passfree_candidates_CNN(module):
    W = module.weight.data
    A = W.abs()
    cands = []
    if isinstance(module, nn.Conv2d):
        OC, IC, KH, KW = W.shape
        flat = A.view(OC, IC, -1)
        vmax, arg = flat.max(dim=2)  # (OC, IC)
        for oc in range(OC):
            for ic in range(IC):
                s = float(vmax[oc, ic])
                if s == 0.0: continue
                p = int(arg[oc, ic])
                kh, kw = divmod(p, KW)
                cands.append((s, (oc, ic, kh, kw), module))
    elif isinstance(module, nn.Linear):
        vmax, arg = A.max(dim=1)     # per output row
        for o in range(W.size(0)):
            s = float(vmax[o])
            if s == 0.0: continue
            j = int(arg[o])
            cands.append((s, (o, j), module))
    return cands

@torch.no_grad()
def flip_sign_(module, index):
    w = module.weight.data
    w[index] = -w[index]


def eval_dnl_ARk(model, loader, L=10, ks=(1,2,3,4,5), device='cuda',
                 collector='passfree'):
    """
    collector: 'passfree' -> use collect_dnl_passfree_candidates_CNN (|w|)
               '1pass'    -> use collect_dnl_passfree_candidates_CNN (needs grads prepared)
    """
    model = model.to(device).eval()
    clean = calc_acc(model, loader, device)

    # choose collector
    if collector == 'passfree':
        get_cands = collect_dnl_passfree_candidates_CNN
    elif collector == '1pass':
        get_cands = collect_dnl_passfree_candidates_CNN
    else:
        raise ValueError("collector must be 'passfree' or '1pass'.")

    # build candidate list from first L param layers
    cands = []
    for m in first_L_param_layers_CNN(model, L):
        cands.extend(get_cands(m))
    cands.sort(key=lambda t: t[0], reverse=True)  # global top by score

    acc_k = {}
    flipped = 0
    for k in ks:
        while flipped < k:
            _, idx, mod = cands[flipped]
            flip_sign_(mod, idx)
            flipped += 1
        acc_k[k] = calc_acc(model, loader, device)

    ar_k = {k: (clean - acc_k[k]) / max(1e-12, clean) for k in ks}
    return clean, acc_k, ar_k, len(cands)

In [31]:
weights = models.ShuffleNet_V2_X1_0_Weights.DEFAULT
model = models.shufflenet_v2_x1_0(weights=weights)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

L = 10
KS = (1,2,3,4,5)
clean, acc_k, ar_k, num_cands = eval_dnl_ARk(model, val_loader, L=L, ks=KS, device=DEVICE)

print("Evaluating pass-free attack on ShuffleNet:")
print(f"Evaluated on {len(val_ds)} images.")
print(f"Baseline top-1 accuracy: {clean:.3f}%")
print(f"Candidates in first {L} layers: {num_cands}")
print("\nAR(k) (cumulative flips):")
print("k\tAcc_k (%)\tAR(k)")
for k in KS:
    print(f"{k}\t{acc_k[k]:.3f}\t\t{ar_k[k]*100:.2f}%")


/home/elgarici-dan/miniconda3/envs/cs236781-hw/lib/python3.8/site-packages/torch/utils/data/dataloader.py:557: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


Evaluating pass-free attack on ShuffleNet:
Evaluated on 50000 images.
Baseline top-1 accuracy: 69.356%
Candidates in first 10 layers: 16452

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	4.210		93.93%
2	0.288		99.58%
3	0.264		99.62%
4	0.216		99.69%
5	0.168		99.76%


### Intermediate Results

As you can see, the DNL pass-free method yields substantial Accuracy Reduction with only few, select weights being flipped.
We reproduced the exact same results from the article displayed in Table 1. This experiment served as a validator that our implemntation is correct and now we can proceed to the more advanced methods.

Now, we will be implementing the DNL 1-pass method, which instead of considering only the magnitude of the paramters, uses an apprixmation of second order taylor expansion of the loss function w.r.t the parameter. So we define the score of a a paramter in this method by:
$Score(\theta_i) = |\theta_i| + \left| \theta_i g_i + \frac{1}{2} \theta_i^2 H_{ii} \right|$

where $ g_i $ denotes the gradient of loss w.r.t the paramter $\theta_i $ and $H_{ii}$ is approximated by $g_i^{2}$

Of course, that in order to compute the gradient we need to have an input we can calculate the loss on, so we define dnl_1pass_prepare_grads
as a utility that generates a random input and makes a single pass through the model.


In [3]:
def dnl_1pass_prepare_grads(model, device='cuda', batch_size=32, H=224, W=224):
    """
    Runs a single forward+backward pass on Gaussian input to populate .grad for all params.
    Uses R(θ) = sum of logits over the batch (Algorithm 2).
    """
    model.to(device).train()  # train/eval doesn't matter for grads; keep BN affine behavior consistent
    for p in model.parameters():
        if p.grad is not None:
            p.grad.zero_()

    X = torch.randn(batch_size, 3, H, W, device=device)  # Gaussian input
    logits = model(X)
    R = logits.sum()  # sum of logits across batch and classes
    R.backward()      # populates .grad for parameters
    model.eval()      # return to eval for accuracy measurements
    
    
def collect_dnl1pass_candidates(module):
    """
    One candidate per 'kernel', scored by the 1P-DNL hybrid score:
      S = |w| + |(w * g) + 0.5 * w^2 * g^2|
    where g = dR/dw, with R = sum of logits on Gaussian input.
    Returns list of tuples (score, index_tuple, module), same as your DNL collector.
    """
    W = module.weight.data
    G = module.weight.grad
    if G is None:
        # user forgot to call dnl_1pass_prepare_grads(model, ...)
        raise RuntimeError("No gradients found on module.weight. Call dnl_1pass_prepare_grads(model, ...) first.")

    # S = |w| + |(w*g) + 0.5*w^2*g^2|  (Gauss–Newton-like diagonal Hessian approx)
    S = W.abs() + ((W * G) + 0.5 * (W * W) * (G * G)).abs()

    cands = []
    if isinstance(module, nn.Conv2d):
        OC, IC, KH, KW = W.shape
        flat = S.view(OC, IC, -1)
        vmax, arg = flat.max(dim=2)  # pick the best element per (out_ch, in_ch) kernel
        for oc in range(OC):
            for ic in range(IC):
                s = float(vmax[oc, ic])
                if s == 0.0: 
                    continue
                p = int(arg[oc, ic])
                kh, kw = divmod(p, KW)
                cands.append((s, (oc, ic, kh, kw), module))
    elif isinstance(module, nn.Linear):
        vmax, arg = S.max(dim=1)     # best column per output row
        OUT = W.size(0)
        for o in range(OUT):
            s = float(vmax[o])
            if s == 0.0:
                continue
            j = int(arg[o])
            cands.append((s, (o, j), module))
    return cands

In [32]:
weights = models.ShuffleNet_V2_X1_0_Weights.DEFAULT
model = models.shufflenet_v2_x1_0(weights=weights)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

L = 10
KS = (1,2,3,4,5)
dnl_1pass_prepare_grads(model, device=DEVICE, batch_size=32, H=224, W=224)
_, acc_k, ar_k, num_cands = eval_dnl_ARk(model, val_loader, L=L, ks=KS, device=DEVICE, collector="1pass")

print("Evaluating 1pass attack on ShuffleNet:")
print(f"Evaluated on {len(val_ds)} images.")
print(f"Baseline top-1 accuracy: {clean:.3f}%")
print(f"Candidates in first {L} layers: {num_cands}")
print("\nAR(k) (cumulative flips):")
print("k\tAcc_k (%)\tAR(k)")
for k in KS:
    print(f"{k}\t{acc_k[k]:.3f}\t\t{ar_k[k]*100:.2f}%")

Evaluating 1pass attack on ShuffleNet:
Evaluated on 50000 images.
Baseline top-1 accuracy: 63.782%
Candidates in first 10 layers: 16452

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	4.306		93.25%
2	0.350		99.45%
3	0.256		99.60%
4	0.180		99.72%
5	0.188		99.71%


## DNL 1-pass Results Analysis
Interestingly, on this particualr model the 1-pass method yields extremly similiar (within 1%) damage, as quantified by the observed AR (Accuracy Reduction), to the more simple pass-free method. This isn't unheard of since as dispalyed in the article in figure 3, while the mean AR of 1P-DNL attack is 10% ~ higher there is high variance in the performance of the methods, and the IQR is around 40 (!)

This makes it feasible that there  exists many models that would perform better or simliarly with pass-free DNL, despite its simplicity.


In [4]:
# !pip install torch torchvision transformers datasets
import os, math, random, torch, torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, logging
from typing import List, Tuple, Dict

logging.set_verbosity_error()   # hide all warnings/errors except fatal

os.environ["TOKENIZERS_PARALLELISM"] = "false"
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42
torch.manual_seed(SEED); random.seed(SEED)

# Part 2: Delving into the realm of LLM
After having reproduced the results from the article, we now wish to extend them to a completely different model architecture.

We will be reusing the same paradigm of only selecting k paramters to sign flip and measuring impact by observing the $AR(k)$ as defined before. This will ensure we are consistent with the research done in the paper and create a shared "lanugage" between the different experiments.

Note that in the realm of LLMs many of the assumptions and tweaks done before and tailored for CNNs no longer apply. For example the paper goes into lengths to describe why we need to choose paramters from disjoint kernels, as flipping signs from the same kernel can negate the effect due to spatial properties that CNN has. All this is lost in the realm of LLM and so we will attempt to pioneer our own herustics and justify them.

Note also that we got used to a very low $k$ generating substantial damage to the model, but LLMs in general have 3 to 4 orders of magnitude more parameters. So we need to recalibrate our expectations on the possible damage a low $k$ can yield, compared to CNNs.

### Setup

As before, to evaluate our methods we will need to choose a concrete model and dataset. In this part we opted to go with BERT and evaluate it on the canoincal SST2 which presents a simple downstream task of sentiment analysis, the model gets text and has to classify it between 2 classes: Positive Sentiment vs Negative Sentiment.

In [5]:
def load_sst2_llm(model_name: str = "textattack/bert-base-uncased-SST-2",
                   batch_size: int = 64,
                   max_train: int = None,
                   max_eval: int = None):
    """
    Loads SST-2 (GLUE) and returns: tokenizer, eval_loader
    (We evaluate on the validation/dev split for AR(k).)
    """
    tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    ds = load_dataset("glue", "sst2")

    # n_params = sum(p.numel() for p in model.parameters())
    # print(f"{n_params:,} parameters")  
    
    def tokenize(batch):
        return tok(batch["sentence"], truncation=True, padding=False, max_length=128)

    # tokenize; keep 'label' by removing only the text column
    ds_tok = ds.map(tokenize, batched=True, remove_columns=["sentence"])

    # some tokenizers return token_type_ids (BERT) and some don't (RoBERTa). It's fine either way.

    # rename 'label' -> 'labels' (what HF models expect for supervised heads)
    if "label" in ds_tok["train"].column_names:
        ds_tok = ds_tok.rename_column("label", "labels")

    # optional subsampling for speed
    def maybe_select(d, n):
        return d.select(range(min(n, len(d)))) if (n is not None) else d

    eval_ds = maybe_select(ds_tok["validation"], max_eval)

    def collate_fn(features):
        # dynamic padding for inputs; labels pass through untouched
        batch = tok.pad(
            {k: [f[k] for f in features] for k in features[0] if k != "labels"},
            return_tensors="pt"
        )
        if "labels" in features[0]:
            batch["labels"] = torch.tensor([f["labels"] for f in features], dtype=torch.long)
        return batch

    eval_loader = DataLoader(
        eval_ds,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=2,
        pin_memory=True
    )
    return tok, eval_loader

def load_bert_sst2_llm(model_name:str="textattack/bert-base-uncased-SST-2"):
    """
    Loads a BERT already fine-tuned on SST-2.
    """
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    model.to(DEVICE)
    model.eval()
    return model

tokenizer_llm, eval_loader_llm = load_sst2_llm(max_eval=None)  # set a number (e.g., 2000) for faster tests


In [6]:
@torch.no_grad()
def accuracy_llm(model, loader, device=DEVICE) -> float:
    model.eval()
    correct = total = 0
    for batch in loader:
        batch = {k: v.to(device, non_blocking=True) for k,v in batch.items()}
        logits = model(**{k:batch[k] for k in ["input_ids","attention_mask"]}).logits
        pred = logits.argmax(-1)
        correct += (pred == batch["labels"]).sum().item()
        total   += batch["labels"].numel()
    return 100.0 * correct / max(1,total)


## BERT Architecture

BERT (Bidirectional Encoder Representations from Transformers) is a Transformer-based language model that learns contextual embeddings for words by looking at both left and right context simultaneously.

At the heart of BERT lies the self-attention mechanism, which allows each token to attend to every other token in the sequence to build rich contextual representations. As we studied in the course attention is defined as:
$
\text{Attention}(Q, K, V) = \text{softmax}\!\left( \frac{QK^{T}}{\sqrt{d_k}} \right) V
$

Where the Q, K and V matrices are learned paramters, and on self-attention more precisely:

 $ Q = W_q * X , K = W_k * X , V = W_v * X $  

where X is the input to the layer.  

We looked at this and tried to pioneer our own methods of "candidate" selection (for sign flip). Due to the large amount of parameters in BERT - over 100 million, we chose to pursue a general strategy of *critical mass*: attempting to focus all our k flips on one mechanism to try and disrupt it.

We hypothesized that since we have limited "budget", our best bet would be to completely collapse a part of the model within that $k$ budget instead of spreading the pressure across the entire model. But before delving more into that, we will present the naive approach which will serve as a baseline. 

### Free for All

First of all, we could simply pick the highest magnitude paramters regardless from which mechanism they're from. This is the most simple method, and its logic is that simply picking the high-impact parameters from everywhere will accumulate "global" damage, disrupting every part of the system. Due to its simplicity, this will serve as our baseline. Since we are not limited in our candidate choice, and we can take from all layers we named it "Free for All"

As before, we will all our experiments on both versions both: pass-free DNL and 1-pass DNL 

In [ ]:
def compatible_with_strategy_conditions(layer_name, strategy) -> bool:
    if strategy is None:
        return True
    if "Q" in strategy and "query" in layer_name:
        return True
    if "K" in strategy and "key" in layer_name:
        return True
    if "V" in strategy and "value" in layer_name:
        return True
    if (".intermediate.dense" in layer_name or ".output.dense" in layer_name) and "FFN" in strategy:
        return True
    if "classifier" in strategy and "classifier" in layer_name:
        return True
    return False    

In [ ]:
def first_L_param_layers_llm(model, L:int, strategy:str or None) -> List[nn.Module]:
    """
    Returns the first L parameter-bearing layers of supported types (nn.Linear, nn.Embedding).
    """
    layers = []
    counter = 0 
    for name, m in model.named_modules():
        if counter >= L:
            break
        # print(name) 
        if isinstance(m, nn.Linear) and compatible_with_strategy_conditions(name, strategy):
            # must have a weight tensor
            if getattr(m, "weight", None) is not None:
                layers.append(m)
                counter += 1
    print("---LOGGING LAYERS----")
    print(f"Layers actually collected: {len(layers)}")
    print("---END LOGGING LAYERS----")

    return layers
    
def eval_dnl_ARk_llm(model,
                     loader,
                     L:int=10,
                     ks:Tuple[int,...]=(1,2,3,4,5),
                     device=DEVICE,
                     collector:str='passfree',
                     strategy: str or None = None,
                     params_geter_function = None):
    """
    collector is either 'passfree','1pass'
    strategy is from {K, Q, V, KQ, KV, QV, KQV, FFN, classifier}
    """
    model.to(device).eval()
    clean = accuracy_llm(model, loader, device)
    params_geter_function = params_geter_function if params_geter_function is not None else first_L_param_layers_llm

    # pick collector
    if collector == 'passfree':
        get_cands = collect_passfree_candidates_llm
    elif collector == '1pass':
        get_cands = collect_1pass_candidates_llm
    else:
        raise ValueError("collector must be 'passfree' or '1pass'.")

    # gather candidates from first L modules
    cands = []
    for m in params_geter_function(model, L, strategy):
        cands.extend(get_cands(m))

    cands.sort(key=lambda t: t[0], reverse=True)
    acc_k = {}
    flipped = 0
    for k in ks:
        while flipped < k:
            if flipped >= len(cands):
                break
            _, idx, module = cands[flipped]
            W = module.weight.data
            # print("---LOGGING W[idx] BEFORE FLIPPING----")
            # print(W[idx])
            flip_sign_llm(module, idx)
            flipped += 1
            # print("---LOGGING W[idx] AFTER FLIPPING----")
            # print(W[idx])
        acc_k[k] = accuracy_llm(model, loader, device)

    ar_k = {k: (clean - acc_k[k]) / max(1e-12, clean) for k in ks}
    return clean, acc_k, ar_k, len(cands)


In [9]:
@torch.no_grad()
def flip_sign_llm(module: nn.Module, index: Tuple[int,int]):
    W = module.weight.data
    W[index] = -W[index]

def dnl_1pass_prepare_grads_llm(model, tokenizer, device=DEVICE, batch_size:int=16, seq_len:int=128):
    """
    Single forward+backward pass on random token sequences to populate parameter grads.
    Uses R(θ) = sum of logits over the batch (like the CNN variant uses sum of logits).
    """
    model.to(device).train()
    for p in model.parameters():
        if p.grad is not None:
            p.grad.zero_()

    # Build random token IDs from tokenizer vocab (avoid special token 0 if it’s [PAD])
    vocab_size = tokenizer.vocab_size
    low_id = 5  # skip very low special tokens
    X = torch.randint(low=low_id, high=vocab_size, size=(batch_size, seq_len), device=device)
    attn = torch.ones_like(X, device=device)

    logits = model(input_ids=X, attention_mask=attn).logits
    R = logits.sum()
    R.backward()
    model.eval()


def collect_passfree_candidates_llm(module: nn.Module) -> List[Tuple[float, Tuple[int, ...], nn.Module]]:
    """
    Collect every scalar parameter in the module's weight tensor.
    Returns a list of (score, index_tuple, module)
      - score: absolute value of weight
      - index_tuple: indices in weight tensor (row, col, ...)
      - module: the layer itself
    """
    cands = []
    if isinstance(module, nn.Linear):
        W = module.weight.data
        A = W.abs()

        # flatten and iterate with unravelled indices
        for flat_idx, val in enumerate(A.view(-1)):
            if val == 0:
                continue
            idx = tuple(torch.unravel_index(torch.tensor(flat_idx), A.shape))
            cands.append((float(val), idx, module))
    return cands

def collect_1pass_candidates_llm(module: nn.Module) -> List[Tuple[float, Tuple[int, int], nn.Module]]:
    """
    One-pass (gradient-informed) candidate scoring:
      S = |w| + (w*g) + 0.5 * w^2 * g^2
    with 'one per kernel' = one element per row.
    """
    cands = []
    if isinstance(module, nn.Linear):
        W = module.weight.data
        G = getattr(module.weight, "grad", None)
        if G is None:
            raise RuntimeError("collect_1pass_candidates_llm: missing grads; call dnl_1pass_prepare_grads_llm first.")
        S = W.abs() + ((W * G) + 0.5 * (W * W) * (G * G)).abs()
        # flatten and iterate with unravelled indices
        for flat_idx, val in enumerate(S.view(-1)):
            if val == 0:
                continue
            idx = tuple(torch.unravel_index(torch.tensor(flat_idx), S.shape))
            cands.append((float(val), idx, module))
    return cands


### Experiment #1 - pass-free DNL attack Free for All


In [9]:
KS = (1,10,100,1_000,10_000, 100_000)
for L in (6, 12, 18): 
    print(f"--- Run Expiriment #1 for L = {L} ---")
    
    model_llm = load_bert_sst2_llm()
    
    # (Optional) quick debug: restrict eval set size earlier with max_eval=2000 in load_sst2_llm
    clean_pf, acck_pf, ark_pf, nC_pf = eval_dnl_ARk_llm(model_llm, eval_loader_llm, L=L, ks=KS, collector='passfree')
    print(ark_pf)
    
    print(f"Evaluating passfree attack on BERT (SST-2):")
    print(f"Evaluated on {len(eval_loader_llm.dataset)} examples.")
    print(f"Baseline top-1 accuracy: {clean_pf:.3f}%")
    print(f"Candidates in first {L} layers: {nC_pf}")
    print("\nAR(k) (cumulative flips):")
    print("k\tAcc_k (%)\tAR(k)")
    
    for k in KS:
        print(f"{k}\t{acck_pf[k]:.3f}\t\t{ark_pf[k]*100:.2f}%")


--- Run Expiriment #1 for L = 6 ---
---LOGGING LAYERS----
Layers actually collected: 6
---END LOGGING LAYERS----
{1: 0.0, 10: 0.0, 100: 0.0012406947890819732, 1000: 0.0012406947890819732, 10000: 0.0012406947890819732, 100000: 0.0024813895781637925}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 6 layers: 7077888

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.317		0.12%
1000	92.317		0.12%
10000	92.317		0.12%
100000	92.202		0.25%
--- Run Expiriment #1 for L = 12 ---
---LOGGING LAYERS----
Layers actually collected: 12
---END LOGGING LAYERS----
{1: 0.0, 10: 0.0, 100: 0.0012406947890819732, 1000: 0.003722084367245766, 10000: 0.008684863523573198, 100000: 0.06079404466501254}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 12 layers: 14155776

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
1

## Experiment #1 results:
This experiment shows just how resilient BERT is to the free for all attack. For example, for $L=6 and k = 100,000$ which represents 1.4% of all parameters in layers 1 through 6, the model is basically unaffected (measly $AR(k) = 0.25%$)

In addition we see that as $L$ (first L layers we choose candiadtes to flip sign from) grows the effect on the model grows substantially. With $L=18$ even $k=1$ already generates $AR(k) = 4%$ which is staggering, considering With $L=12$ even $k=100000$ only yields $AR(k) = 6%$

### Intermediate conclusion 
The basic conclusion here is that the first layers matter less when attacking LLMs. In CNNs the effect was the exact opposite - attacking the L last layers yielded neglible impact compared to attacking the first L layers with the same L, this effect can be seen clearly in Table 1 in the article. These results raised the possiblity that perhaps the last layers have a larger signficiance in BERTs architecture, and we'll revisit this hypothesis later.

Additionally, we reach some kind of sturation as $k$ increases, $k=1000$ gives us a "local" maximum and then increasing k by 2 orders of magnitude only improves $AR(k)$ by less than 2 points. The same effect was seen in CNNs althoguh with much lower k. In the article, we see that on ShuffleNetV2 k=2 and k=5 end up doing almost the same in the model, so this effect is reproduced here.

Even still, when taking L=18 which covers almost 3/4 of BERT's layers and taking a whopping $k = 100,000$ we endup with $AR(k) = 44.79%$ which serves as a strong indicator of how much more resilient BERT is to this kind of attack compared to CNNs.

### Experiment #2 - 1-pass DNL attack Free for All

In [9]:
KS = (1,10,100,1_000,10_000, 100_000)
for L in (6, 12, 18): 
    print(f"--- Run Expiriment #2 for L = {L} ---")
    
    model_llm = load_bert_sst2_llm()

    dnl_1pass_prepare_grads_llm(model_llm, tokenizer_llm, device=DEVICE, batch_size=16, seq_len=128)
    # (Optional) quick debug: restrict eval set size earlier with max_eval=2000 in load_sst2_llm
    clean_pf, acck_pf, ark_pf, nC_pf = eval_dnl_ARk_llm(model_llm, eval_loader_llm, L=L, ks=KS, collector='1pass')
    
    print(f"Evaluating 1pass attack on BERT (SST-2):")
    print(f"Evaluated on {len(eval_loader_llm.dataset)} examples.")
    print(f"Baseline top-1 accuracy: {clean_pf:.3f}%")
    print(f"Candidates in first {L} layers: {nC_pf}")
    print("\nAR(k) (cumulative flips):")
    print("k\tAcc_k (%)\tAR(k)")
    for k in KS:
        print(f"{k}\t{acck_pf[k]:.3f}\t\t{ark_pf[k]*100:.2f}%")

--- Run Expiriment #2 for L = 6 ---
---LOGGING LAYERS----
Layers actually collected: 6
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 6 layers: 7077888

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.317		0.12%
1000	92.202		0.25%
10000	92.661		-0.25%
100000	91.628		0.87%
--- Run Expiriment #2 for L = 12 ---
---LOGGING LAYERS----
Layers actually collected: 12
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 12 layers: 14155776

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.317		0.12%
1000	92.546		-0.12%
10000	92.087		0.37%
100000	82.110		11.17%
--- Run Expiriment #2 for L = 18 ---
---LOGGING LAYERS----
Layers actually collected: 18
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 exam

## Experiment #2 results:

Next to damage when targeting the first layers, and curiously even improving the model (!) on this specific task of SST-2 with combination of $k=10,000 L = 6$. We believe this in part due to the fact BERT wasn't actually trained on SST-2.

When going deeper $L=12$ barely any difference except for largest $k$. This hints to us the highest score paramters, which in this case is gradient based, are probably also the highest magnitude paramters which explains the lack of difference between the results to pass-free.

When going extremly deep $L=18$ we see how the optimum point (in terms of damage to th model) is achieved with relatively low $k = 1000$ which is an interesting phenonmonon that wasn't captured when attacking CNNs. 

## Expanding to other attack strategies

As discussed before we wanted to move beyond the "free for all" variant, and try a more sophisticated approach that is based on the architecture of the model.

What if instead of distributing the damage across the model we focus on one component and try to push it to its breaking point? 

This line lead us to come up with the approach of focusing on one "type" of weights at a time from the attention mechanism. In the following segments we will run experiments which will focus only on K, Q, and V and then different combinations of them to see what is most effective at harming the model.


In this segment we'll run a considerable amount of experiments and then aggregate at the end our conclusions.



In [ ]:
import json 

def run_experiment(KS_list, LS_list, exp_num, exp_collector, exp_strategy = None, params_geter_function = None):
    experiment_data = {}
    for L in LS_list: 
        print(f"--- Run Experiment #{exp_num} for L = {L} ---")
        
        model_llm = load_bert_sst2_llm()
        
        # (Optional) quick debug: restrict eval set size earlier with max_eval=2000 in load_sst2_llm
        clean_pf, acck_pf, ark_pf, nC_pf = eval_dnl_ARk_llm(model_llm, eval_loader_llm, L=L, ks=KS, collector=exp_collector, strategy = exp_strategy, params_geter_function = params_geter_function)
        experiment_data[L] = ark_pf
        print(ark_pf)
        
        print(f"Evaluating {exp_collector} attack on BERT (SST-2):")
        print(f"Evaluated on {len(eval_loader_llm.dataset)} examples.")
        print(f"Baseline top-1 accuracy: {clean_pf:.3f}%")
        print(f"Candidates in first {L} layers: {nC_pf}")
        print("\nAR(k) (cumulative flips):")
        print("k\tAcc_k (%)\tAR(k)")
        
        for k in KS:
            print(f"{k}\t{acck_pf[k]:.3f}\t\t{ark_pf[k]*100:.2f}%")

    with open("/home/elgarici-dan/project_deep/experiment_log.json", "r") as f:
        log_data = json.load(f)
    
    log_data[exp_num] = experiment_data
    
    with open("/home/elgarici-dan/project_deep/experiment_log.json", "w") as f:
        json.dump(log_data, f)

## Expirment #3 - pass free attack on Q layers only.

In this expriment we only flip signs of parameters from the Queries component in the attention mechanism.

In [ ]:
KS = (1,10,100,1_000,10_000, 100_000)
LS = (1, 3, 6, 12)
run_experiment(KS, LS, exp_num = 3, exp_collector = "passfree", exp_strategy= "Q")

--- Run Expiriment #3 for L = 1 ---
---LOGGING LAYERS----
Layers actually collected: 1
---END LOGGING LAYERS----
{1: 0.0, 10: 0.0, 100: 0.0012406947890819732, 1000: 0.0024813895781637925, 10000: 0.004962779156327585, 100000: 0.006203473945409559}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 1 layers: 589824

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.317		0.12%
1000	92.202		0.25%
10000	91.972		0.50%
100000	91.858		0.62%
--- Run Expiriment #3 for L = 3 ---
---LOGGING LAYERS----
Layers actually collected: 3
---END LOGGING LAYERS----
{1: 0.0, 10: 0.0, 100: 0.0, 1000: 0.003722084367245766, 10000: 0.0074441687344913784, 100000: 0.03722084367245658}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 3 layers: 1769472

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.4

## Expirment #4 - pass free attack on K layers only.
In this expriment we only flip signs of parameters from the Keys component in the attention mechanism.

In [ ]:
KS = (1,10,100,1_000,10_000, 100_000)
LS = (1, 3, 6, 12)
run_experiment(KS, LS, exp_num = 4, exp_collector = "passfree", exp_strategy= "K")

--- Run Expiriment #4 for L = 1 ---
---LOGGING LAYERS----
Layers actually collected: 1
---END LOGGING LAYERS----
{1: 0.0, 10: 0.0, 100: 0.0, 1000: -0.0012406947890818195, 10000: -0.002481389578163639, 100000: 0.0024813895781637925}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 1 layers: 589824

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.546		-0.12%
10000	92.661		-0.25%
100000	92.202		0.25%
--- Run Expiriment #4 for L = 3 ---
---LOGGING LAYERS----
Layers actually collected: 3
---END LOGGING LAYERS----
{1: 0.0, 10: 0.0, 100: 0.0, 1000: 0.0, 10000: 0.003722084367245766, 100000: 0.03101736972704718}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 3 layers: 1769472

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.431		0.00%
10

## Expirment #5 - pass free attack on V layers only.
In this expriment we only flip signs of parameters from the Value component in the attention mechanism.

In [ ]:
KS = (1,10,100,1_000,10_000, 100_000)
LS = (1, 3, 6, 12)
run_experiment(KS, LS, exp_num = 5, exp_collector = "passfree", exp_strategy= "V")

--- Run Expiriment #5 for L = 1 ---
---LOGGING LAYERS----
Layers actually collected: 1
---END LOGGING LAYERS----
{1: 0.0, 10: 0.0, 100: 0.0, 1000: 0.0, 10000: 0.003722084367245766, 100000: 0.01116625310173699}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 1 layers: 589824

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.431		0.00%
10000	92.087		0.37%
100000	91.399		1.12%
--- Run Expiriment #5 for L = 3 ---
---LOGGING LAYERS----
Layers actually collected: 3
---END LOGGING LAYERS----
{1: 0.0, 10: 0.0, 100: 0.0012406947890819732, 1000: -0.0012406947890818195, 10000: -0.0012406947890818195, 100000: 0.026054590570719745}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 3 layers: 1769472

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.317		0.12%
1000	

## Expirment #6 - pass free attack on QK layers.
In this expriment we only flip signs of parameters from the Queries and Key components in the attention mechanism.

In [ ]:
KS = (1,10,100,1_000,10_000, 100_000)
LS = (2, 4, 8)
run_experiment(KS, LS, exp_num = 6, exp_collector = "passfree", exp_strategy= "QK")

--- Run Expiriment #6 for L = 2 ---
---LOGGING LAYERS----
Layers actually collected: 2
---END LOGGING LAYERS----
{1: 0.0, 10: 0.0, 100: 0.0, 1000: 0.0024813895781637925, 10000: 0.003722084367245766, 100000: -0.0012406947890818195}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 2 layers: 1179648

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.202		0.25%
10000	92.087		0.37%
100000	92.546		-0.12%
--- Run Expiriment #6 for L = 4 ---
---LOGGING LAYERS----
Layers actually collected: 4
---END LOGGING LAYERS----
{1: 0.0, 10: 0.0, 100: 0.0, 1000: 0.003722084367245766, 10000: 0.003722084367245766, 100000: 0.0074441687344913784}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 4 layers: 2359296

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
100

## Expirment #7 - pass free attack on QV layers.
In this expriment we only flip signs of parameters from the Queries and Values components in the attention mechanism.

In [ ]:
KS = (1,10,100,1_000,10_000, 100_000)
LS = (2, 4, 8)
run_experiment(KS, LS, exp_num = 7, exp_collector = "passfree", exp_strategy= "QV")

--- Run Expiriment #7 for L = 2 ---
---LOGGING LAYERS----
Layers actually collected: 2
---END LOGGING LAYERS----
{1: 0.0, 10: 0.0, 100: 0.0012406947890819732, 1000: 0.0012406947890819732, 10000: 0.0012406947890819732, 100000: 0.006203473945409559}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 2 layers: 1179648

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.317		0.12%
1000	92.317		0.12%
10000	92.317		0.12%
100000	91.858		0.62%
--- Run Expiriment #7 for L = 4 ---
---LOGGING LAYERS----
Layers actually collected: 4
---END LOGGING LAYERS----
{1: 0.0, 10: 0.0, 100: 0.0012406947890819732, 1000: 0.0012406947890819732, 10000: 0.008684863523573198, 100000: 0.013647642679900783}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 4 layers: 2359296

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	9

## Expirment #8 - pass free attack on KV layers.
In this expriment we only flip signs of parameters from the Value and Key components in the attention mechanism.

In [ ]:
KS = (1,10,100,1_000,10_000, 100_000)
LS = (2, 4, 8)
run_experiment(KS, LS, exp_num = 8, exp_collector = "passfree", exp_strategy= "KV")

--- Run Expiriment #8 for L = 2 ---
---LOGGING LAYERS----
Layers actually collected: 2
---END LOGGING LAYERS----
{1: 0.0, 10: 0.0, 100: 0.0, 1000: -0.0012406947890818195, 10000: 0.0, 100000: 0.0024813895781637925}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 2 layers: 1179648

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.546		-0.12%
10000	92.431		0.00%
100000	92.202		0.25%
--- Run Expiriment #8 for L = 4 ---
---LOGGING LAYERS----
Layers actually collected: 4
---END LOGGING LAYERS----
{1: 0.0, 10: 0.0, 100: 0.0, 1000: 0.006203473945409559, 10000: 0.0074441687344913784, 100000: 0.003722084367245766}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 4 layers: 2359296

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	91.858		0.62%
1

## Expirment #9 - pass free attack on KQV layers.
In this expriment we only flip signs of parameters from the Queries, Keys and Values components, basically ignoring the MLP head and embedding layers.

In [ ]:
KS = (1,10,100,1_000,10_000, 100_000)
LS = (3, 6, 9)
run_experiment(KS, LS, exp_num = 9, exp_collector = "passfree", exp_strategy= "KQV")

--- Run Expiriment #9 for L = 3 ---
---LOGGING LAYERS----
Layers actually collected: 3
---END LOGGING LAYERS----
{1: 0.0, 10: 0.0, 100: 0.0, 1000: 0.0024813895781637925, 10000: 0.003722084367245766, 100000: -0.003722084367245612}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 3 layers: 1769472

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.202		0.25%
10000	92.087		0.37%
100000	92.775		-0.37%
--- Run Expiriment #9 for L = 6 ---
---LOGGING LAYERS----
Layers actually collected: 6
---END LOGGING LAYERS----
{1: 0.0, 10: 0.0, 100: 0.0, 1000: 0.003722084367245766, 10000: 0.00992555831265517, 100000: 0.008684863523573198}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 6 layers: 3538944

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	9

## Expirment #10 - pass free attack on FFN layers.
In this expriment we only flip signs of parameters from the Feed-forward network.

In [ ]:
KS = (1,10,100,1_000,10_000, 100_000)
LS = (2, 4, 8)
run_experiment(KS, LS, exp_num = 10, exp_collector = "passfree", exp_strategy= "FFN")

##todo: Rerun this experiment with different L
## there is 0 cands in first layers since ffn is only at the end of the model

--- Run Expiriment #10 for L = 2 ---
---LOGGING LAYERS----
Layers actually collected: 0
---END LOGGING LAYERS----
{1: 0.0, 10: 0.0, 100: 0.0, 1000: 0.0, 10000: 0.0, 100000: 0.0}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 2 layers: 0

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.431		0.00%
10000	92.431		0.00%
100000	92.431		0.00%
--- Run Expiriment #10 for L = 4 ---
---LOGGING LAYERS----
Layers actually collected: 0
---END LOGGING LAYERS----
{1: 0.0, 10: 0.0, 100: 0.0, 1000: 0.0, 10000: 0.0, 100000: 0.0}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 4 layers: 0

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.431		0.00%
10000	92.431		0.00%
100000	92.431		0.00%
--- Run Expiriment #10 for L = 8 ---
---LOGGING LAYERS----
L

## Expirment #11 - pass free attack on the classifier layer.
In this expriment we only flip signs of parameters from the MLP head of the model

In [ ]:
KS = (1,10,100,500,1_000, 2_000)
LS = (1,)
run_experiment(KS, LS, exp_num = 11, exp_collector = "passfree", exp_strategy= "classifier")

--- Run Expiriment #11 for L = 1 ---
---LOGGING LAYERS----
Layers actually collected: 1
---END LOGGING LAYERS----
{1: 0.0, 10: 0.0024813895781637925, 100: 0.0074441687344913784, 500: 0.9168734491315136, 1000: 0.9168734491315136, 2000: 0.9181141439205956}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 1 layers: 1536

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.202		0.25%
100	91.743		0.74%
500	7.683		91.69%
1000	7.683		91.69%
2000	7.569		91.81%


# Phase 2 - 1-pass DNL using novel attack strategies

In this segement we'll rerun our previous experiments this time using the more sohpisticated, gradient based 1-pass strategy discussed in the article.

In [ ]:
def run_expiriment_one_pass(KS_list, LS_list, exp_num, exp_strategy = None, params_geter_function = None):
    experiment_data = {}
    for L in LS_list:
        print(f"--- Run Expiriment #{exp_num} for L = {L} ---")
        
        model_llm = load_bert_sst2_llm()
    
        dnl_1pass_prepare_grads_llm(model_llm, tokenizer_llm, device=DEVICE, batch_size=16, seq_len=128)
        # (Optional) quick debug: restrict eval set size earlier with max_eval=2000 in load_sst2_llm
        clean_pf, acck_pf, ark_pf, nC_pf = eval_dnl_ARk_llm(model_llm, eval_loader_llm, L=L, ks=KS, collector='1pass', strategy = exp_strategy, params_geter_function = params_geter_function)
        experiment_data[L] = ark_pf

        print(f"Evaluating 1pass attack on BERT (SST-2):")
        print(f"Evaluated on {len(eval_loader_llm.dataset)} examples.")
        print(f"Baseline top-1 accuracy: {clean_pf:.3f}%")
        print(f"Candidates in first {L} layers: {nC_pf}")
        print("\nAR(k) (cumulative flips):")
        print("k\tAcc_k (%)\tAR(k)")
        for k in KS_list:
            print(f"{k}\t{acck_pf[k]:.3f}\t\t{ark_pf[k]*100:.2f}%")

    with open("/home/elgarici-dan/project_deep/experiment_log.json", "r") as f:
        log_data = json.load(f)
    
    log_data[exp_num] = experiment_data
    
    with open("/home/elgarici-dan/project_deep/experiment_log.json", "w") as f:
        json.dump(log_data, f)

# Expirment #12 - 1-pass attack on the K layer.
In this expriment we only flip signs of parameters from the Keys component in the attention mechanism.

In [ ]:
KS = (1,10,100,1_000,10_000, 100_000)
LS = (1, 3, 6, 12)
run_expiriment_one_pass(KS, LS, exp_num = 12, exp_strategy= "K")

--- Run Expiriment #12 for L = 1 ---
---LOGGING LAYERS----
Layers actually collected: 1
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 1 layers: 589824

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.546		-0.12%
10000	92.546		-0.12%
100000	92.317		0.12%
--- Run Expiriment #12 for L = 3 ---
---LOGGING LAYERS----
Layers actually collected: 3
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 3 layers: 1769472

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.317		0.12%
10000	92.087		0.37%
100000	89.794		2.85%
--- Run Expiriment #12 for L = 6 ---
---LOGGING LAYERS----
Layers actually collected: 6
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.

# Expirment #13 - 1pass attack on the Q layer.
we tryed difrent values of Q.

In [ ]:
KS = (1,10,100,1_000,10_000, 100_000)
LS = (1, 3, 6, 12)
run_expiriment_one_pass(KS, LS, exp_num = 13, exp_strategy= "Q")

--- Run Expiriment #13 for L = 1 ---
---LOGGING LAYERS----
Layers actually collected: 1
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 1 layers: 589824

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.317		0.12%
1000	92.317		0.12%
10000	92.087		0.37%
100000	91.743		0.74%
--- Run Expiriment #13 for L = 3 ---
---LOGGING LAYERS----
Layers actually collected: 3
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 3 layers: 1769472

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.317		0.12%
10000	91.743		0.74%
100000	88.991		3.72%
--- Run Expiriment #13 for L = 6 ---
---LOGGING LAYERS----
Layers actually collected: 6
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
B

# Expirment #14 - 1pass attack on the V layer.
we tryed difrent values of V.

In [ ]:

KS = (1,10,100,1_000,10_000, 100_000)
LS = (1, 3, 6, 12)
run_expiriment_one_pass(KS, LS, exp_num = 14, exp_strategy= "V")

--- Run Expiriment #14 for L = 1 ---
---LOGGING LAYERS----
Layers actually collected: 1
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 1 layers: 589824

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.431		0.00%
10000	92.087		0.37%
100000	91.628		0.87%
--- Run Expiriment #14 for L = 3 ---
---LOGGING LAYERS----
Layers actually collected: 3
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 3 layers: 1769472

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.317		0.12%
10	92.317		0.12%
100	92.661		-0.25%
1000	92.317		0.12%
10000	92.775		-0.37%
100000	89.106		3.60%
--- Run Expiriment #14 for L = 6 ---
---LOGGING LAYERS----
Layers actually collected: 6
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.

# Expirment #15 - 1pass attack on the K,Q layer.
we tryed difrent values of K.

In [ ]:
KS = (1,10,100,1_000,10_000, 100_000)
LS = (2, 4, 8)
run_expiriment_one_pass(KS, LS, exp_num = 15, exp_strategy= "KQ")

--- Run Expiriment #15 for L = 2 ---
---LOGGING LAYERS----
Layers actually collected: 2
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 2 layers: 1179648

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.202		0.25%
10000	92.202		0.25%
100000	92.317		0.12%
--- Run Expiriment #15 for L = 4 ---
---LOGGING LAYERS----
Layers actually collected: 4
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 4 layers: 2359296

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.087		0.37%
10000	91.858		0.62%
100000	91.858		0.62%
--- Run Expiriment #15 for L = 8 ---
---LOGGING LAYERS----
Layers actually collected: 8
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.


# Expirment #16 - 1pass attack on the KV layer.
we tryed difrent values of K.

In [ ]:
KS = (1,10,100,1_000,10_000, 100_000)
LS = (2,4,8)
run_expiriment_one_pass(KS, LS, exp_num = 16, exp_strategy= "KV")

--- Run Expiriment #16 for L = 2 ---
---LOGGING LAYERS----
Layers actually collected: 2
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 2 layers: 1179648

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.546		-0.12%
10000	92.546		-0.12%
100000	92.431		0.00%
--- Run Expiriment #16 for L = 4 ---
---LOGGING LAYERS----
Layers actually collected: 4
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 4 layers: 2359296

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.202		0.25%
1000	91.858		0.62%
10000	91.858		0.62%
100000	92.202		0.25%
--- Run Expiriment #16 for L = 8 ---
---LOGGING LAYERS----
Layers actually collected: 8
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples

# Experiment #17 - 1pass attack on the QV layer.
we tryed difrent values of K.

In [ ]:
KS = (1,10,100,1_000,10_000, 100_000)
LS = (2,4,8)
run_expiriment_one_pass(KS, LS, exp_num = 17, exp_strategy= "QV")

--- Run Expiriment #17 for L = 2 ---
---LOGGING LAYERS----
Layers actually collected: 2
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 2 layers: 1179648

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.317		0.12%
1000	92.317		0.12%
10000	92.202		0.25%
100000	91.858		0.62%
--- Run Expiriment #17 for L = 4 ---
---LOGGING LAYERS----
Layers actually collected: 4
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 4 layers: 2359296

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.317		0.12%
1000	92.431		0.00%
10000	91.858		0.62%
100000	91.284		1.24%
--- Run Expiriment #17 for L = 8 ---
---LOGGING LAYERS----
Layers actually collected: 8
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.


# Experiment #18 - 1pass attack on the KQV layer.
we tryed difrent values of K.

In [ ]:
KS = (1,10,100,1_000,10_000, 100_000)
LS = (3, 6, 9)
run_expiriment_one_pass(KS, LS, exp_num = 18, exp_strategy= "KQV")

--- Run Expiriment #18 for L = 3 ---
---LOGGING LAYERS----
Layers actually collected: 3
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 3 layers: 1769472

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.202		0.25%
10000	91.972		0.50%
100000	92.661		-0.25%
--- Run Expiriment #18 for L = 6 ---
---LOGGING LAYERS----
Layers actually collected: 6
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 6 layers: 3538944

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.087		0.37%
10000	91.284		1.24%
100000	91.858		0.62%
--- Run Expiriment #18 for L = 9 ---
---LOGGING LAYERS----
Layers actually collected: 9
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.

# Experiment #19 - 1pass attack on the FFN layer.
we tryed difrent values of K.

In [ ]:
KS = (1,10,100,1_000,10_000, 100_000)
LS = (2,4,8)
run_expiriment_one_pass(KS, LS, exp_num = 19, exp_strategy= "FFN")

--- Run Expiriment #19 for L = 2 ---
---LOGGING LAYERS----
Layers actually collected: 0
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 2 layers: 0

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.431		0.00%
10000	92.431		0.00%
100000	92.431		0.00%
--- Run Expiriment #19 for L = 4 ---
---LOGGING LAYERS----
Layers actually collected: 0
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 4 layers: 0

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.431		0.00%
10000	92.431		0.00%
100000	92.431		0.00%
--- Run Expiriment #19 for L = 8 ---
---LOGGING LAYERS----
Layers actually collected: 0
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top

# Experiment #20 - 1pass attack on the classifier layer.
we tryed difrent values of K.

In [ ]:
KS = (1,10,100,500,1_000, 2_000)
LS = (1,)
run_expiriment_one_pass(KS, LS, exp_num = 20 , exp_strategy= "classifier")

--- Run Expiriment #20 for L = 1 ---
---LOGGING LAYERS----
Layers actually collected: 1
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 1 layers: 1536

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.202		0.25%
100	92.317		0.12%
500	7.798		91.56%
1000	7.913		91.44%
2000	7.569		91.81%


In [ ]:
def last_L_param_layers_llm(model, L:int, strategy:str or None) -> List[nn.Module]:
    """
    Returns the first L parameter-bearing layers of supported types (nn.Linear, nn.Embedding).
    """
    layers = []
    counter = 0 
    for name, m in reversed(list(model.named_modules())):
        if counter >= L:
            break
        # print(name) 
        if isinstance(m, nn.Linear) and compatible_with_strategy_conditions(name, strategy):
            # must have a weight tensor
            if getattr(m, "weight", None) is not None:
                layers.append(m)
                counter += 1
    return layers

# Experiment 21 - revers order of layers

In [ ]:
KS = (1,10,100,1_000,10_000, 100_000)
LS = (6, 12, 18)
run_experiment(KS, LS, exp_num = 21, exp_collector = "passfree", exp_strategy = None, params_geter_function = last_L_param_layers_llm)

--- Run Experiment #21 for L = 6 ---


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

{1: 0.0, 10: 0.0, 100: 0.0, 1000: 0.0012406947890819732, 10000: 0.0024813895781637925, 100000: 0.003722084367245766}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 6 layers: 6489600

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.317		0.12%
10000	92.202		0.25%
100000	92.087		0.37%
--- Run Experiment #21 for L = 12 ---
{1: 0.0, 10: 0.0, 100: 0.0, 1000: 0.0012406947890819732, 10000: 0.0074441687344913784, 100000: 0.006203473945409559}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 12 layers: 13567488

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.317		0.12%
10000	91.743		0.74%
100000	91.858		0.62%
--- Run Experiment #21 for L = 18 ---
{1: 0.0, 10: 0.0012406947890819732, 100: 0.0, 1000: 0.05210918114143934, 10000: 0.4317617866

# Experiment 22 - revers order of layers, only Q layers

In [ ]:
KS = (1,10,100,1_000,10_000, 100_000)
LS = (1, 3, 6, 12)
run_experiment(KS, LS, exp_num = 22, exp_collector = "passfree", exp_strategy= "Q", params_geter_function = last_L_param_layers_llm)

--- Run Experiment #22 for L = 1 ---
{1: 0.0, 10: 0.0, 100: 0.0, 1000: 0.0024813895781637925, 10000: 0.0024813895781637925, 100000: 0.01116625310173699}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 1 layers: 589824

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.202		0.25%
10000	92.202		0.25%
100000	91.399		1.12%
--- Run Experiment #22 for L = 3 ---
{1: 0.0, 10: 0.0, 100: 0.0, 1000: 0.006203473945409559, 10000: 0.0024813895781637925, 100000: 0.01861042183622837}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 3 layers: 1769472

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	91.858		0.62%
10000	92.202		0.25%
100000	90.711		1.86%
--- Run Experiment #22 for L = 6 ---
{1: 0.0, 10: 0.0, 100: 0.0, 1000: 0.004962779156327585, 10000: 

# Experiment 23 - revers order of layers, only V layers

In [ ]:
KS = (1,10,100,1_000,10_000, 100_000)
LS = (1, 3, 6, 12)
run_experiment(KS, LS, exp_num = 23, exp_collector = "passfree", exp_strategy= "V", params_geter_function = last_L_param_layers_llm)

--- Run Experiment #23 for L = 1 ---
{1: 0.0, 10: 0.0, 100: 0.0, 1000: 0.0, 10000: 0.0012406947890819732, 100000: 0.02109181141439216}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 1 layers: 589824

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.431		0.00%
10000	92.317		0.12%
100000	90.482		2.11%
--- Run Experiment #23 for L = 3 ---
{1: 0.0, 10: 0.0, 100: 0.0024813895781637925, 1000: 0.004962779156327585, 10000: 0.003722084367245766, 100000: 0.014888337468982757}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 3 layers: 1769472

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.202		0.25%
1000	91.972		0.50%
10000	92.087		0.37%
100000	91.055		1.49%
--- Run Experiment #23 for L = 6 ---
{1: 0.0, 10: 0.0, 100: 0.0012406947890819732, 1000: 0.002481389

# Experiment 24 - revers order of layers, only K layers

In [ ]:
KS = (1,10,100,1_000,10_000, 100_000)
LS = (1, 3, 6, 12)
run_experiment(KS, LS, exp_num = 24, exp_collector = "passfree", exp_strategy= "K", params_geter_function = last_L_param_layers_llm)

--- Run Experiment #24 for L = 1 ---
{1: 0.0, 10: 0.0, 100: 0.0, 1000: 0.0012406947890819732, 10000: 0.0, 100000: 0.0074441687344913784}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 1 layers: 589824

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.317		0.12%
10000	92.431		0.00%
100000	91.743		0.74%
--- Run Experiment #24 for L = 3 ---
{1: 0.0, 10: 0.0, 100: 0.0024813895781637925, 1000: 0.006203473945409559, 10000: 0.008684863523573198, 100000: 0.01861042183622837}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 3 layers: 1769472

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.202		0.25%
1000	91.858		0.62%
10000	91.628		0.87%
100000	90.711		1.86%
--- Run Experiment #24 for L = 6 ---
{1: 0.0, 10: 0.003722084367245766, 100: 0.013647642679900783, 

# Experiment 25 - revers order of layers, only Q,K layers

In [ ]:
KS = (1,10,100,1_000,10_000, 100_000)
LS = (2, 4, 8)
run_experiment(KS, LS, exp_num = 25, exp_collector = "passfree", exp_strategy= "QK", params_geter_function = last_L_param_layers_llm)

--- Run Experiment #25 for L = 2 ---
{1: 0.0, 10: 0.0, 100: 0.0, 1000: 0.0024813895781637925, 10000: 0.0024813895781637925, 100000: 0.006203473945409559}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 2 layers: 1179648

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.202		0.25%
10000	92.202		0.25%
100000	91.858		0.62%
--- Run Experiment #25 for L = 4 ---
{1: 0.0, 10: 0.0, 100: 0.0, 1000: 0.003722084367245766, 10000: 0.003722084367245766, 100000: 0.003722084367245766}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 4 layers: 2359296

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.087		0.37%
10000	92.087		0.37%
100000	92.087		0.37%
--- Run Experiment #25 for L = 8 ---
{1: 0.0, 10: 0.0, 100: 0.003722084367245766, 1000: 0.00372208

# Experiment 26 - revers order of layers, only Q,V layers

In [ ]:
KS = (1,10,100,1_000,10_000, 100_000)
LS = (2, 4, 8)
run_experiment(KS, LS, exp_num = 25, exp_collector = "passfree", exp_strategy= "QV", params_geter_function = last_L_param_layers_llm)

--- Run Experiment #25 for L = 2 ---
{1: 0.0, 10: 0.0, 100: 0.0, 1000: 0.0024813895781637925, 10000: 0.0012406947890819732, 100000: 0.006203473945409559}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 2 layers: 1179648

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.202		0.25%
10000	92.317		0.12%
100000	91.858		0.62%
--- Run Experiment #25 for L = 4 ---
{1: 0.0, 10: 0.0, 100: 0.0, 1000: 0.0024813895781637925, 10000: 0.004962779156327585, 100000: 0.0024813895781637925}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 4 layers: 2359296

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.202		0.25%
10000	91.972		0.50%
100000	92.202		0.25%
--- Run Experiment #25 for L = 8 ---
{1: 0.0, 10: 0.0, 100: 0.0, 1000: -0.0012406947890818195, 1

# Experiment 27 - revers order of layers, only K,V layers

In [ ]:
KS = (1,10,100,1_000,10_000, 100_000)
LS = (2, 4, 8)
run_experiment(KS, LS, exp_num = 27, exp_collector = "passfree", exp_strategy= "KV", params_geter_function = last_L_param_layers_llm)

--- Run Experiment #27 for L = 2 ---
{1: 0.0, 10: 0.0, 100: 0.0, 1000: 0.0012406947890819732, 10000: 0.0012406947890819732, 100000: 0.0074441687344913784}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 2 layers: 1179648

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.317		0.12%
10000	92.317		0.12%
100000	91.743		0.74%
--- Run Experiment #27 for L = 4 ---
{1: 0.0, 10: 0.0, 100: 0.0, 1000: 0.0024813895781637925, 10000: 0.003722084367245766, 100000: 0.006203473945409559}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 4 layers: 2359296

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.202		0.25%
10000	92.087		0.37%
100000	91.858		0.62%
--- Run Experiment #27 for L = 8 ---
{1: 0.0, 10: 0.0, 100: 0.004962779156327585, 1000: 0.007444

# Experiment 28 - revers order of layers, only K,Q, V layers

In [ ]:
KS = (1,10,100,1_000,10_000, 100_000)
LS = (3, 6, 9)
run_experiment(KS, LS, exp_num = 28, exp_collector = "passfree", exp_strategy= "KQV", params_geter_function = last_L_param_layers_llm)

--- Run Experiment #28 for L = 3 ---
{1: 0.0, 10: 0.0, 100: 0.0, 1000: 0.0024813895781637925, 10000: 0.0024813895781637925, 100000: 0.0012406947890819732}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 3 layers: 1769472

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.202		0.25%
10000	92.202		0.25%
100000	92.317		0.12%
--- Run Experiment #28 for L = 6 ---
{1: 0.0, 10: 0.0, 100: 0.0, 1000: 0.0024813895781637925, 10000: 0.006203473945409559, 100000: 0.006203473945409559}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 6 layers: 3538944

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.202		0.25%
10000	91.858		0.62%
100000	91.858		0.62%
--- Run Experiment #28 for L = 9 ---
{1: 0.0, 10: 0.0, 100: 0.0012406947890819732, 1000: 0.00620

# Expirment #29 - 1pass attack on the K layers.
we tryed difrent values of K.

In [ ]:
KS = (1,10,100,1_000,10_000, 100_000)
LS = (1, 3, 6, 12)
run_expiriment_one_pass(KS, LS, exp_num = 29, exp_strategy= "K", params_geter_function = last_L_param_layers_llm)

--- Run Expiriment #29 for L = 1 ---
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 1 layers: 589824

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.317		0.12%
10000	92.317		0.12%
100000	91.743		0.74%
--- Run Expiriment #29 for L = 3 ---
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 3 layers: 1769472

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.202		0.25%
1000	91.858		0.62%
10000	91.628		0.87%
100000	90.711		1.86%
--- Run Expiriment #29 for L = 6 ---
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 6 layers: 3538944

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.317		0.12%
100	91.743		0.74%
1000	90.940		1.61%
10000	91.284		1.24%
100000	89.450		3.23%
--- R

# Expirment #30 - 1pass attack on the Q layers.
we tryed difrent values of K.

In [ ]:
KS = (1,10,100,1_000,10_000, 100_000)
LS = (1, 3, 6, 12)
run_expiriment_one_pass(KS, LS, exp_num = 30, exp_strategy= "Q", params_geter_function = last_L_param_layers_llm)

--- Run Expiriment #30 for L = 1 ---
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 1 layers: 589824

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.202		0.25%
10000	92.202		0.25%
100000	91.284		1.24%
--- Run Expiriment #30 for L = 3 ---
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 3 layers: 1769472

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	91.858		0.62%
10000	92.202		0.25%
100000	90.826		1.74%
--- Run Expiriment #30 for L = 6 ---
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 6 layers: 3538944

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.317		0.12%
1000	91.972		0.50%
10000	91.284		1.24%
100000	91.055		1.49%
--- R

# Expirment #31 - 1pass attack on the V layers.
we tryed difrent values of K.

In [ ]:
KS = (1,10,100,1_000,10_000, 100_000)
LS = (1, 3, 6, 12)
run_expiriment_one_pass(KS, LS, exp_num = 31, exp_strategy= "V", params_geter_function = last_L_param_layers_llm)

--- Run Expiriment #31 for L = 1 ---
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 1 layers: 589824

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.431		0.00%
10000	92.431		0.00%
100000	90.940		1.61%
--- Run Expiriment #31 for L = 3 ---
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 3 layers: 1769472

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.317		0.12%
1000	92.087		0.37%
10000	92.087		0.37%
100000	88.761		3.97%
--- Run Expiriment #31 for L = 6 ---
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 6 layers: 3538944

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.317		0.12%
100	92.202		0.25%
1000	92.202		0.25%
10000	91.514		0.99%
100000	88.073		4.71%
--- R

# Expirment #32 - 1pass attack on the K,Q layers.
we tryed difrent values of K.

In [ ]:
KS = (1,10,100,1_000,10_000, 100_000)
LS = (2, 4, 8)
run_expiriment_one_pass(KS, LS, exp_num = 32, exp_strategy= "KQ", params_geter_function = last_L_param_layers_llm)

--- Run Expiriment #32 for L = 2 ---
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 2 layers: 1179648

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.202		0.25%
10000	92.087		0.37%
100000	91.972		0.50%
--- Run Expiriment #32 for L = 4 ---
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 4 layers: 2359296

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.087		0.37%
10000	92.087		0.37%
100000	92.202		0.25%
--- Run Expiriment #32 for L = 8 ---
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 8 layers: 4718592

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.087		0.37%
1000	91.972		0.50%
10000	91.743		0.74%
100000	89.679		2.98%


# Expirment #33 - 1pass attack on the K,V layers.
we tryed difrent values of K.

In [ ]:
KS = (1,10,100,1_000,10_000, 100_000)
LS = (2, 4, 8)
run_expiriment_one_pass(KS, LS, exp_num = 33, exp_strategy= "KV", params_geter_function = last_L_param_layers_llm)

--- Run Expiriment #33 for L = 2 ---
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 2 layers: 1179648

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.431		0.00%
10000	92.202		0.25%
100000	91.628		0.87%
--- Run Expiriment #33 for L = 4 ---
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 4 layers: 2359296

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.202		0.25%
10000	92.202		0.25%
100000	91.743		0.74%
--- Run Expiriment #33 for L = 8 ---
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 8 layers: 4718592

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.202		0.25%
100	92.087		0.37%
1000	91.628		0.87%
10000	91.628		0.87%
100000	90.023		2.61%


# Expirment #34 - 1pass attack on the Q,V layers.
we tryed difrent values of K.

In [ ]:
KS = (1,10,100,1_000,10_000, 100_000)
LS = (2, 4, 8)
run_expiriment_one_pass(KS, LS, exp_num = 34, exp_strategy= "QV", params_geter_function = last_L_param_layers_llm)

--- Run Expiriment #34 for L = 2 ---
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 2 layers: 1179648

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.317		0.12%
10000	92.317		0.12%
100000	91.628		0.87%
--- Run Expiriment #34 for L = 4 ---
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 4 layers: 2359296

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.317		0.12%
1000	92.202		0.25%
10000	92.087		0.37%
100000	91.514		0.99%
--- Run Expiriment #34 for L = 8 ---
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 8 layers: 4718592

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.431		0.00%
10000	92.202		0.25%
100000	89.794		2.85%


# Expirment #35 - 1pass attack on the K,Q,V layers.
we tryed difrent values of K.

In [ ]:
KS = (1,10,100,1_000,10_000, 100_000)
LS = (3, 6, 9)
run_expiriment_one_pass(KS, LS, exp_num = 35, exp_strategy= "KQV", params_geter_function = last_L_param_layers_llm)

--- Run Expiriment #35 for L = 3 ---
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 3 layers: 1769472

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.202		0.25%
10000	92.202		0.25%
100000	92.202		0.25%
--- Run Expiriment #35 for L = 6 ---
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 6 layers: 3538944

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.202		0.25%
10000	92.202		0.25%
100000	91.514		0.99%
--- Run Expiriment #35 for L = 9 ---
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 9 layers: 5308416

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.317		0.12%
1000	91.858		0.62%
10000	91.743		0.74%
100000	90.138		2.48%


# Old and not relevante!

In [23]:
import re
import torch
import torch.nn as nn
from typing import List, Tuple, Dict

def _role_from_name_llm(name: str, module: nn.Module) -> str:
    """
    Classify BERT module roles by dotted path:
      Q/K/V, AttnOut, FFN1 (intermediate), FFN2 (output), CLS (classifier), EMB (embeddings), LN (layernorm)
    """
    if isinstance(module, nn.Linear):
        if ".attention.self.query" in name:   return "Q"
        elif ".attention.self.key"   in name:   return "K"
        elif ".attention.self.value" in name:   return "V"
        elif ".attention.output.dense" in name: return "AttnOut"
        # FFN parts (exclude attention output)
        elif ".intermediate.dense" in name:               return "FFN1"
        elif ".output.dense" in name and ".attention." not in name: return "FFN2"
        elif name.endswith("classifier"):       return "CLS"
    if isinstance(module, nn.Embedding):
        return "EMB"
    if isinstance(module, nn.LayerNorm):
        return "LN"
    return "OTHER"

def _iter_param_modules_llm(model: nn.Module) -> List[Tuple[str, nn.Module, str]]:
    """Yield (name, module, role) for parameter-bearing modules we care about."""
    for name, m in model.named_modules():
        if isinstance(m, (nn.Linear, nn.Embedding)):
            if getattr(m, "weight", None) is not None:
                yield name, m, _role_from_name_llm(name, m)

def _select_modules_by_strategy_llm(model: nn.Module, strategy: str, L: int) -> List[nn.Module]:
    """
    Filter parameter modules by strategy, preserve forward order, then keep first L.
      strategies:
        - 'attn_Q_only' / 'attn_K_only' / 'attn_V_only'
        - 'attn_QKV_interlace' (handled specially in collector; here we pass Q+K+V)
        - 'mlp_only'  (FFN1 + FFN2)
        - 'classifier_head_only' (CLS)
    """
    roles_keep = set()
    if strategy == "attn_Q_only": roles_keep = {"Q"}
    elif strategy == "attn_K_only": roles_keep = {"K"}
    elif strategy == "attn_V_only": roles_keep = {"V"}
    elif strategy == "attn_QKV_interlace": roles_keep = {"Q","K","V"}
    elif strategy == "mlp_only": roles_keep = {"FFN1","FFN2"}
    elif strategy == "classifier_head_only": roles_keep = {"CLS"}
    else:
        raise ValueError(f"Unknown strategy '{strategy}'")

    kept = []
    for name, m, role in _iter_param_modules_llm(model):
        if role in roles_keep:
            kept.append(m)
    return kept[:L]


In [24]:
def _rowwise_argmax_candidates_llm(score_tensor: torch.Tensor, module: nn.Module):
    """
    From score tensor shaped like weight (rows x cols), select 1 element per row with max score.
    Returns list[(score, (row, col), module)].
    """
    vmax, arg = score_tensor.max(dim=1)
    cands = []
    rows = score_tensor.size(0)
    for r in range(rows):
        s = float(vmax[r])
        if s == 0.0:
            continue
        j = int(arg[r])
        cands.append((s, (r, j), module))
    return cands

def collect_passfree_candidates_llm_filtered(modules: List[nn.Module]):
    """Pass-free: |w| per row (one-per-kernel)."""
    out = []
    for m in modules:
        W = m.weight.data
        A = W.abs()
        out.extend(_rowwise_argmax_candidates_llm(A, m))
    return out

def collect_1pass_candidates_llm_filtered(modules: List[nn.Module]):
    """1-pass: S = |w| + w*g + 0.5*w^2*g^2 per row (one-per-kernel)."""
    out = []
    for m in modules:
        W = m.weight.data
        G = getattr(m.weight, "grad", None)
        if G is None:
            raise RuntimeError("collect_1pass_candidates_llm_filtered: grads missing. Call dnl_1pass_prepare_grads_llm first.")
        S = W.abs() + (W * G) + 0.5 * (W * W) * (G * G)
        out.extend(_rowwise_argmax_candidates_llm(S, m))
    return out


In [25]:
def collect_interlace_QKV_candidates_llm(model: nn.Module, L: int, mode: str = "pass_free"):
    """
    Build three candidate lists (Q, K, V), sort each by score desc, then round-robin merge.
    mode: 'pass_free' or '1pass'
    """
    # pick modules for each role, honoring first-L within each role in forward order
    mods_Q, mods_K, mods_V = [], [], []
    for name, m, role in _iter_param_modules_llm(model):
        if role == "Q": mods_Q.append(m)
        elif role == "K": mods_K.append(m)
        elif role == "V": mods_V.append(m)
    mods_Q, mods_K, mods_V = mods_Q[:L], mods_K[:L], mods_V[:L]

    if mode == "pass_free":
        build = collect_passfree_candidates_llm_filtered
    elif mode == "1pass":
        build = collect_1pass_candidates_llm_filtered
    else:
        raise ValueError("mode must be 'pass_free' or '1pass'")

    cq = sorted(build(mods_Q), key=lambda t: t[0], reverse=True)
    ck = sorted(build(mods_K), key=lambda t: t[0], reverse=True)
    cv = sorted(build(mods_V), key=lambda t: t[0], reverse=True)

    # round-robin merge
    merged, i, j, k = [], 0, 0, 0
    while i < len(cq) or j < len(ck) or k < len(cv):
        if i < len(cq): 
            merged.append(cq[i]); i += 1
        if j < len(ck): 
            merged.append(ck[j]); j += 1
        if k < len(cv): 
            merged.append(cv[k]); k += 1
    return merged


In [26]:
def eval_dnl_ARk_llm_strategy(model,
                              loader,
                              tokenizer,
                              L:int=10,
                              ks:Tuple[int,...]=(1,2,3,4,5),
                              device=DEVICE,
                              mode:str="pass_free",
                              strategy:str="attn_Q_only"):
    """
    mode: 'pass_free' or '1pass'
    strategy: one of
      'attn_Q_only','attn_K_only','attn_V_only',
      'attn_QKV_interlace','mlp_only','classifier_head_only'
    """
    model.to(device).eval()

    # 1) clean accuracy
    clean = accuracy_llm(model, loader, device)

    # 2) prepare grads if 1-pass
    if mode == "1pass":
        dnl_1pass_prepare_grads_llm(model, tokenizer, device=device, batch_size=16, seq_len=128)

    # 3) build candidate list according to strategy
    if strategy == "attn_QKV_interlace":
        cands = collect_interlace_QKV_candidates_llm(model, L=L, mode=("1pass" if mode=="1pass" else "pass_free"))
    else:
        modules = _select_modules_by_strategy_llm(model, strategy=strategy, L=L)
        if mode == "pass_free":
            cands = collect_passfree_candidates_llm_filtered(modules)
        else:
            cands = collect_1pass_candidates_llm_filtered(modules)

        cands.sort(key=lambda t: t[0], reverse=True)

    # 4) cumulative flips + eval
    acc_k = {}
    flipped = 0
    for k in ks:
        while flipped < k and flipped < len(cands):
            _, idx, mod = cands[flipped]
            flip_sign_llm(mod, idx)
            flipped += 1
        acc_k[k] = accuracy_llm(model, loader, device)

    ar_k = {k: (clean - acc_k[k]) / max(1e-12, clean) for k in ks}
    return clean, acc_k, ar_k, len(cands)


In [27]:
def print_results_strategy_llm(attack_name:str,
                               clean:float,
                               acc_k:Dict[int,float],
                               ar_k:Dict[int,float],
                               num_cands:int,
                               L:int,
                               KS:Tuple[int,...],
                               eval_loader):
    print(f"Evaluating {attack_name} attack on BERT:")
    print(f"Evaluated on {len(eval_loader.dataset)} samples.")
    print(f"Baseline top-1 accuracy: {clean:.3f}%")
    print(f"Candidates in first {L} layers: {num_cands}")
    print("\nAR(k) (cumulative flips):")
    print("k\tAcc_k (%)\tAR(k)")
    for k in KS:
        print(f"{k}\t{acc_k[k]:.3f}\t\t{ar_k[k]*100:.2f}%")


# Conclusions

As we can see, LLMs are more resilient in the fact of such attacks, and the plethora of methods we tried don't yield simliar $AR(k)$ as seen in CNNs. A priori, the most straight forward explanation comes from the difference in scale - BERT has 4 orders of magintude more paramters than most CNNs, but it doesn't end there. Even when we adjusted $k$ accordingly and we flipped proportionally same amount of parameters we still couldn't reach levels of damage that were possible while attacking CNNs.

Another thing we discovered is that our hypothesis was wrong - due to the way the attention mechanism works the attacks that yielded the best results were distributing the damage:
selecting candidates from both Q and V or K and V. Selecting Q and K or focusing only at one at a time yielded inferior results.

----------------------------------------

## Utilities / Helpers

In [29]:
for name, m, role in _iter_param_modules_llm(model_llm):
    print(f"Cur layer module is: {name} m = {m} role = {role}")


Cur layer module is: bert.embeddings.word_embeddings m = Embedding(30522, 768, padding_idx=0) role = EMB
Cur layer module is: bert.embeddings.position_embeddings m = Embedding(512, 768) role = EMB
Cur layer module is: bert.embeddings.token_type_embeddings m = Embedding(2, 768) role = EMB
Cur layer module is: bert.encoder.layer.0.attention.self.query m = Linear(in_features=768, out_features=768, bias=True) role = Q
Cur layer module is: bert.encoder.layer.0.attention.self.key m = Linear(in_features=768, out_features=768, bias=True) role = K
Cur layer module is: bert.encoder.layer.0.attention.self.value m = Linear(in_features=768, out_features=768, bias=True) role = V
Cur layer module is: bert.encoder.layer.0.attention.output.dense m = Linear(in_features=768, out_features=768, bias=True) role = AttnOut
Cur layer module is: bert.encoder.layer.0.intermediate.dense m = Linear(in_features=768, out_features=3072, bias=True) role = FFN1
Cur layer module is: bert.encoder.layer.0.output.dense m =